# Imports and paths

In [6]:
from pathlib import Path
import os
import torch
import gymnasium as gym
import ale_py

from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

# Register Atari/ALE environments like ALE/Pong-v5
gym.register_envs(ale_py)

# Figure out project root whether notebook runs from /code or project root
cwd = Path.cwd()

if (cwd / "models").exists() and (cwd / "videos").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "models").exists() and (cwd.parent / "videos").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

CODE_DIR = PROJECT_ROOT / "code"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
VIDEO_DIR = PROJECT_ROOT / "videos"

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
VIDEO_DIR.mkdir(exist_ok=True)

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)
print("Video directory:", VIDEO_DIR)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Current working directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\code
Project root: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3
Model directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\models
Results directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\results
Video directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\videos
Torch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


# Create a small A2C Pong environment

## Fresh training only

Run the next environment/model/training-loop cells only if starting a new A2C training run from scratch.

In [8]:
ENV_ID = "ALE/Pong-v5"
SEED = 0
#N_ENVS = 4
N_ENVS = 16

env = make_atari_env(
    ENV_ID,
    n_envs=N_ENVS,
    seed=SEED,
    wrapper_kwargs=dict(terminal_on_life_loss=False)
)

env = VecFrameStack(env, n_stack=4)

obs = env.reset()

print("Environment:", ENV_ID)
print("Observation shape:", obs.shape)
print("Number of environments:", N_ENVS)

Environment: ALE/Pong-v5
Observation shape: (16, 84, 84, 4)
Number of environments: 16


# Create a small A2C model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = A2C(
    "CnnPolicy",
    env,
    verbose=1,
    learning_rate=7e-4,
    gamma=0.99,
    device=device,
)

print("A2C model created.")
print("Model device:", model.device)

Using cuda device
Wrapping the env in a VecTransposeImage.
A2C model created.
Model device: cuda


In [ ]:
from stable_baselines3 import A2C
from stable_baselines3.common.sb2_compat.rmsprop_tf_like import RMSpropTFLike

device = "cuda" if torch.cuda.is_available() else "cpu"
# tunned model
model = A2C(
    "CnnPolicy",
    env,
    verbose=1,
    learning_rate=7e-4,
    n_steps=20,
    gamma=0.99,
    gae_lambda=1.0,
    ent_coef=0.01,
    vf_coef=0.25,
    max_grad_norm=0.5,
    normalize_advantage=True,
    policy_kwargs=dict(
        optimizer_class=RMSpropTFLike,
        optimizer_kwargs=dict(eps=1e-5),
    ),
    device=device,
)

print("A2C model created.")
print("Model device:", model.device)

Using cuda device
Wrapping the env in a VecTransposeImage.
A2C model created.
Model device: cuda


In [9]:
from stable_baselines3 import A2C
from stable_baselines3.common.sb2_compat.rmsprop_tf_like import RMSpropTFLike

device = "cuda" if torch.cuda.is_available() else "cpu"
# tunned model2

model = A2C(
    "CnnPolicy",
    env,
    verbose=1,

    # Atari/RL-Zoo-style settings
    learning_rate=7e-4,
    n_steps=5,
    gamma=0.99,
    gae_lambda=1.0,
    ent_coef=0.01,
    vf_coef=0.25,
    max_grad_norm=0.5,
    normalize_advantage=True,

    policy_kwargs=dict(
        optimizer_class=RMSpropTFLike,
        optimizer_kwargs=dict(eps=1e-5),
    ),

    device=device,
)

Using cuda device
Wrapping the env in a VecTransposeImage.


# Small checkpoint training loop

First loop is mainly for a fresh training run.

Example:

* Train from 0 → 10k.
* Save.
* Continue same in-memory model 10k → 20k.
* Save.
* Continue same in-memory model 20k → 30k.
* Save.

In [ ]:
# Small test checkpoints
#checkpoint_steps = [10_000, 20_000, 30_000]
#checkpoint_steps = [100_000, 250_000, 500_000]
#checkpoint_steps = [100_000, 250_000, 500_000, 1_000_000]
checkpoint_steps = [
    10_000,
    20_000,
    30_000,
    40_000,
    50_000,
    150_000,
    250_000,
    350_000,
    500_000,
    1_000_000,
    1_500_000,
    2_000_000,
    3_000_000,
    4_000_000,
    5_000_000,
]
previous_steps = 0

for target_steps in checkpoint_steps:
    steps_to_train = target_steps - previous_steps
    
    print("=" * 60)
    print(f"Training from {previous_steps:,} to {target_steps:,} timesteps")
    print(f"Training for {steps_to_train:,} new timesteps")
    print("=" * 60)
    
    model.learn(
        total_timesteps=steps_to_train,
        reset_num_timesteps=False
    )
    
    checkpoint_path = MODEL_DIR / f"a2c_pong_{target_steps}.zip"
    model.save(checkpoint_path)
    
    print(f"Saved checkpoint: {checkpoint_path}")
    
    previous_steps = target_steps

print("A2C small checkpoint test complete.")

Training from 0 to 10,000 timesteps
Training for 10,000 new timesteps
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 884       |
|    ep_rew_mean        | -20.6     |
| time/                 |           |
|    fps                | 494       |
|    iterations         | 100       |
|    time_elapsed       | 16        |
|    total_timesteps    | 8000      |
| train/                |           |
|    entropy_loss       | -1.79     |
|    explained_variance | -0.0228   |
|    learning_rate      | 0.0007    |
|    n_updates          | 99        |
|    policy_loss        | -0.000912 |
|    value_loss         | 0.354     |
-------------------------------------
Saved checkpoint: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\models\a2c_pong_10000.zip
Training from 10,000 to 20,000 timesteps
Training for 10,000 new timesteps
------------------------------------
| rollout/              |          |
|    ep_len_

# Close environment after training

In [5]:
env.close()
print("Training environment closed.")

Training environment closed.


In [1]:
from pathlib import Path
import os
import torch
import gymnasium as gym
import ale_py

from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

# Register Atari/ALE environments like ALE/Pong-v5
gym.register_envs(ale_py)

# Figure out project root whether notebook runs from /code or project root
cwd = Path.cwd()

if (cwd / "models").exists() and (cwd / "videos").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "models").exists() and (cwd.parent / "videos").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

CODE_DIR = PROJECT_ROOT / "code"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
VIDEO_DIR = PROJECT_ROOT / "videos"

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
VIDEO_DIR.mkdir(exist_ok=True)

ENV_ID = "ALE/Pong-v5"
SEED = 0
N_ENVS = 4

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)
print("Video directory:", VIDEO_DIR)
print("Environment:", ENV_ID)
print("Number of environments:", N_ENVS)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Current working directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\code
Project root: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3
Model directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\models
Results directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\results
Video directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\videos
Environment: ALE/Pong-v5
Number of environments: 4
Torch version: 2.11.0+cu128
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


# Reload a checkpoint

## Continue training from a saved checkpoint

If reopening the notebook later, run the setup cell at the top first, then start here.

In [2]:
# Pick the checkpoint you want to continue from
resume_from_steps = 2_000_000

# How much to train before saving each new checkpoint
extra_training_steps = 1_000_000

# How many times to repeat the extra training
num_resume_runs = 3

reload_path = MODEL_DIR / f"a2c_pong_{resume_from_steps}.zip"

if not reload_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {reload_path}")

print("Loading checkpoint:", reload_path)

# Recreate the same kind of environment used for training
resume_env = make_atari_env(
    ENV_ID,
    n_envs=N_ENVS,
    seed=SEED,
    wrapper_kwargs=dict(terminal_on_life_loss=False)
)

resume_env = VecFrameStack(resume_env, n_stack=4)

# Load selected checkpoint
loaded_model = A2C.load(
    reload_path,
    env=resume_env,
    device=device
)

print("Loaded model successfully.")
print("Loaded model device:", loaded_model.device)

Loading checkpoint: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\models\a2c_pong_2000000.zip
Wrapping the env in a VecTransposeImage.
Loaded model successfully.
Loaded model device: cuda


# Resume training from loaded checkpoint

In [3]:
current_steps = resume_from_steps

for run_index in range(1, num_resume_runs + 1):
    next_steps = current_steps + extra_training_steps

    print("=" * 60)
    print(f"Resume run {run_index}/{num_resume_runs}")
    print(f"Training from {current_steps:,} to {next_steps:,} timesteps")
    print(f"Training for {extra_training_steps:,} additional timesteps")
    print("=" * 60)

    loaded_model.learn(
        total_timesteps=extra_training_steps,
        reset_num_timesteps=False
    )

    checkpoint_path = MODEL_DIR / f"a2c_pong_{next_steps}.zip"
    loaded_model.save(checkpoint_path)

    print(f"Saved checkpoint: {checkpoint_path}")

    current_steps = next_steps

resume_env.close()
print("Resume environment closed.")
print("Resume checkpoint loop complete.")

Resume run 1/3
Training from 2,000,000 to 3,000,000 timesteps
Training for 1,000,000 additional timesteps
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.29e+03 |
|    ep_rew_mean        | -20.1    |
| time/                 |          |
|    fps                | 315      |
|    iterations         | 100      |
|    time_elapsed       | 25       |
|    total_timesteps    | 2008000  |
| train/                |          |
|    entropy_loss       | -0.448   |
|    explained_variance | 0.217    |
|    learning_rate      | 0.0007   |
|    n_updates          | 25099    |
|    policy_loss        | -0.0492  |
|    value_loss         | 0.464    |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.32e+03 |
|    ep_rew_mean        | -20.1    |
| time/                 |          |
|    fps                | 308      |
|    iterations         | 200      |
|    t